# Raw Ingestion

## Import Helper Functions

In [1]:
import boto3
import hashlib
import subprocess
from pathlib import Path

from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config_loader import load_config
# from src.spark_sql_magic import sql
# from src.file_utils import checksum
# from src.storage_utils import get_spark_path
# from src.metadata_utils import ddl_metadata_table, already_ingested, log_ingestion_metadata

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
RAW_FOLDER = cfg["raw"]["raw_folder"]
RAW_METADATA_TABLE = cfg["raw"]["raw_metadata_table"]
BUCKET = cfg["general"]["bucket"]
RAW_DATA_DIR = cfg["raw"]["raw_ingestion_path"]
ALLOWED_TYPES = set(cfg["raw"]["allowed_types"])

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import *
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["raw_ingestion"])
        .getOrCreate()
)

## DDL: raw_ingestion_metadata

In [4]:
table_name = f"{CATALOG}.{RAW_FOLDER}.{RAW_METADATA_TABLE}"

ddl_metadata_table(spark, table_name)

polaris.raw.raw_ingestion_metadata created.


## Create s3 Client (Minio)

In [5]:
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)

## Function: Upload Raw File

In [6]:
def upload_raw_file(file_path:str) -> str:
    """
    Upload file to MinIO and log metadata to Iceberg.
    Preserves original format.
    """
    
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    load_date = datetime.date.today().strftime("%Y-%m-%d")

    # central dedup check
    file_checksum = checksum(file_path)
    if already_ingested(spark, table_name, file_checksum):
        print(f"⚠️ {file_path.name} already ingested. Skipping.")
        return get_spark_path(file_path, load_date, BUCKET, RAW_FOLDER, ALLOWED_TYPES)

    # Upload to MinIO (raw format preserved)
    object_key = f"{RAW_FOLDER}/{file_path.name}"

    s3.upload_file(
        str(file_path),
        BUCKET,
        object_key
    )
    spark_path = f"s3a://{BUCKET}/{object_key}"

    log_ingestion_metadata(
        spark,
        table_name,
        file_path,
        spark_path,
        checksum(file_path),
        ALLOWED_TYPES
    )

    print(f"✅ {file_path.name} ingested.")

    return spark_path

In [7]:
raw_data_dir = Path(RAW_DATA_DIR)
files = list(raw_data_dir.glob("*"))

if not files:
    print("No files found.")
else:
    uploaded_paths = []

    for file in files:
        spark_path = upload_raw_file(file)
        uploaded_paths.append(spark_path)

    print("\nUploaded paths:")
    for p in uploaded_paths:
        print(p)

polaris.raw.raw_ingestion_metadata ingested
✅ olist_customers_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_geolocation_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_orders_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_order_items_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_order_payments_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_order_reviews_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_products_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ olist_sellers_dataset.csv ingested.
polaris.raw.raw_ingestion_metadata ingested
✅ product_category_name_translation.csv ingested.

Uploaded paths:
s3a://olist-ecommerce/raw/olist_customers_dataset.csv
s3a://olist-ecommerce/raw/olist_geolocation_dataset.csv
s3a://olist-ecommerce/raw/olist_orders_dataset.csv
s3a://olist-ecommerce/raw/olist_order_items_datas

## Sanity Check

In [8]:
spark.sql(f" FROM {table_name} ").show()

+--------------------+---------+--------------------+----------+--------------------+--------------------+
|           file_name|file_type|                path|size_bytes|           upload_ts|            checksum|
+--------------------+---------+--------------------+----------+--------------------+--------------------+
|olist_orders_data...|      csv|s3a://olist-ecomm...|  17654914|2026-03-05 16:31:...|8bd60e55c1ca229d9...|
|olist_geolocation...|      csv|s3a://olist-ecomm...|  61273883|2026-03-05 16:31:...|6d8464e41c8e20139...|
|olist_order_items...|      csv|s3a://olist-ecomm...|  15438671|2026-03-05 16:31:...|f4fa76976662e9cba...|
|olist_customers_d...|      csv|s3a://olist-ecomm...|   9033957|2026-03-05 16:30:...|8a2c4244856aab4bd...|
|olist_order_payme...|      csv|s3a://olist-ecomm...|   5777138|2026-03-05 16:31:...|75ce0c041d18e5f25...|
|olist_products_da...|      csv|s3a://olist-ecomm...|   2379446|2026-03-05 16:31:...|e935401104a366922...|
|olist_order_revie...|      csv|s3a:/

In [9]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  